# Tập 1.6 · Tìm đường bằng may rủi: RRT 🌳

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap16/tap16_rrt.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 1 · ⏱ ~40 phút · Giáo trình: Principles of Robot Autonomy, [Chương 4](https://porabook.com/book/motion-planning/) (Mục 4.4.2)

Gieo điểm ngẫu nhiên, mọc cây tìm đường. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video (RRT và RRT*).
2. **Tự viết** RRT (3 bài), có ô chấm tự động.
3. Vẽ cây của bạn cạnh cây RRT*.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap16")
sys.path[:0] = ["bai_tap", "sim", ".."]

import rrt as goc                  # mô phỏng trong video
from ve_do_thi import ve           # vẽ cây
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
print(f"RRT : tìm thấy đường sau {goc.RRT['vong_tim_thay']} vòng, cây {len(goc.RRT['V'])} đỉnh, đường dài {goc.RRT['dai']:.2f} m")
for n in (300, 800, 1500, 2500):
    print(f"RRT*: sau {n:4d} vòng, đường tốt nhất dài {goc.dai_sau(goc.RRT_SAO, n):.2f} m")

## 2. Bài tập

Mỗi vòng RRT: gieo $q_{rand}$ (10% gieo thẳng vào đích) → tìm $q_{near}$ gần nhất trên cây → tiến 0,5 m về phía $q_{rand}$ được $q_{new}$ → không xuyên tường thì gắn vào cây → đủ gần đích thì dừng.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `gan_nhat(V, q)` | chỉ số đỉnh gần q nhất |
| **2** | `tien_ve(q_near, q_rand, buoc)` | điểm cách q_near đúng `buoc` về phía q_rand |
| **3** | `rrt(seed, toi_da)` | vòng lặp RRT (docstring ghi thứ tự gọi `rng` để ra đúng số trong video) |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math
import random

from rrt import ALPHA, BUOC, DIEM_DICH, H, Q_G, Q_S, W, khong_va_cham  # noqa: F401  (bản đồ + kiểm tra va chạm có sẵn, trong ../sim)


def gan_nhat(V, q):
    """Bài 1: chỉ số của đỉnh trong danh sách V gần điểm q nhất (khoảng cách Euclid)."""
    raise NotImplementedError("TODO Bài 1: trả về chỉ số i")


def tien_ve(q_near, q_rand, buoc):
    """Bài 2: từ q_near tiến về phía q_rand một đoạn dài `buoc` (nếu q_rand gần hơn `buoc` thì tới luôn q_rand).

    Gợi ý: d = khoảng cách, s = min(1, buoc/d), q_new = q_near + s·(q_rand − q_near). Trả về tuple (x, y)."""
    raise NotImplementedError("TODO Bài 2: trả về q_new")


def rrt(seed=2, toi_da=3000):
    """Bài 3: vòng lặp RRT (Mục 4.4.2 của sách). Dùng rng = random.Random(seed) và gọi rng ĐÚNG thứ tự này để ra giống video:

    Bắt đầu: V = [Q_S], cha = [-1]. Mỗi vòng k = 0, 1, 2, ...:
      1. Lấy mẫu: nếu rng.random() < ALPHA thì q_rand = Q_G (thiên về đích),
         ngược lại q_rand = (rng.uniform(0, W), rng.uniform(0, H)).
      2. i_near = gan_nhat(V, q_rand); nếu q_rand trùng hẳn V[i_near] (khoảng cách < 1e-9) thì bỏ qua vòng này.
      3. q_new = tien_ve(V[i_near], q_rand, BUOC). Nếu q_new ra ngoài không gian hoặc
         khong_va_cham(V[i_near], q_new) là False thì bỏ qua.
      4. Thêm q_new vào V, cha của nó là i_near.
      5. Nếu q_new cách Q_G dưới DIEM_DICH thì dừng: đã tìm thấy đường ở vòng k + 1.
    Trả về dict {"V": V, "cha": cha, "duong": [Q_S, ..., q_new, Q_G], "vong_tim_thay": k + 1}
    (duong lấy bằng cách lần theo cha từ q_new về Q_S rồi đảo ngược; không tìm thấy thì duong = [], vong_tim_thay = None)."""
    raise NotImplementedError("TODO Bài 3: vòng lặp RRT")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Đổi xác suất gieo vào đích (goal bias) rồi xem RRT tìm ra đường nhanh hay chậm (trung bình 10 lần chạy).

In [ ]:
import loi_giai
for alpha in [0.0, 0.1, 0.3, 0.6]:
    loi_giai.ALPHA = alpha
    vong = [loi_giai.rrt(seed=200 + s, toi_da=4000)["vong_tim_thay"] or 4000 for s in range(10)]
    print(f"goal bias {alpha:.1f}: trung bình {sum(vong) / len(vong):6.0f} vòng mới tìm ra đường")
loi_giai.ALPHA = goc.ALPHA

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap16).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)